# New All Analysis CBE

This notebook consolidates the main downstream analyses and figures for the new CBE validation screen.

Scope:
- counts QC
- average editing overview
- editing distributions and PAM comparisons
- LFC correlation across tissues and baselines
- hit overlap comparisons
- gene-level waterfall plots

Primary source data come from the validation screen analysis in the sibling repository:
`../PhD-FSR-MH-Lab/07_B-ALL_resubmission_20250819/validation_screen_analysis`


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import scipy.stats

from sklearn.decomposition import PCA
try:
    from matplotlib_venn import venn2
except ModuleNotFoundError:
    venn2 = None

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['text.usetex'] = False
sns.set_style('white')


In [ ]:
ROOT = Path.cwd()
VAL_DIR = ROOT.parent / 'PhD-FSR-MH-Lab' / '07_B-ALL_resubmission_20250819' / 'validation_screen_analysis'
FIG_DIR = ROOT / 'figures' / 'new_cbe_analysis'
FIG_DIR.mkdir(parents=True, exist_ok=True)

assert VAL_DIR.exists(), f'Validation directory not found: {VAL_DIR}'

LIB = pd.read_csv(VAL_DIR / 'FINAL_focused_library_PAM.csv', index_col=0)
LIB_OG = pd.read_csv(VAL_DIR / 'MBESv2_CORRECTED_PAM.csv', index_col=0)
COSMIC = pd.read_csv(ROOT / 'source-data' / 'Census_allSun Nov 17 02_26_47 2024.csv').fillna('Undefined')

CBE = LIB[LIB['Editor'] == 'CBE'].copy()
CBE_OG_LIB = LIB_OG[(LIB_OG['Editor'] == 'CBE') & (LIB_OG['classification'] == 'targeting guide')].copy()
LIST_NGG_PAM = [f'{x}GG' for x in ['A', 'T', 'C', 'G']]
CBE_NGG = CBE[CBE['PAM_NGN'].isin(LIST_NGG_PAM)].copy()
GUIDES_NGG_CBE_FOCUSED = set(CBE_NGG['gRNA_id'])
GUIDES_NGG_CBE_OG = set(CBE_OG_LIB[CBE_OG_LIB['PAM_NGN'].isin(LIST_NGG_PAM)]['gRNA_id'])

CBE_BC_COUNTS = pd.read_csv(VAL_DIR / 'CBE_BC_COUNTS.txt', sep='\t')
CBE_EPO_COUNTS = pd.read_csv(VAL_DIR / 'CBE_EPO_COUNTS.txt', sep='\t')

LFC_CBE = pd.read_csv(VAL_DIR / 'CBE_LFC_FDR_df.csv')
LFC_CBE_VS_BC = pd.read_csv(VAL_DIR / 'CBE_LFC_FDR_df_vs_bc.csv')
LFC_CBE_OG = pd.read_csv(VAL_DIR / 'CBE_LFC_FDR_df_og.csv')

CRISPRESSO_DIR = VAL_DIR / 'crispresso_data' / 'compact_lane_merged_unfiltered'
MLE_DIR = VAL_DIR / 'MLE'

name_dict = {
    'lib': 'Plasmid',
    'input': 'Input (D0)',
    'd5': 'D5',
    'd15': 'D15',
    'spleen': 'Spleen',
    'bm': 'Bone Marrow',
    'men': 'Meninges',
    'bonemarrow': 'Bone Marrow',
    'meninges': 'Meninges'
}

palette_group = {
    'Plasmid': 'black',
    'Input (D0)': sns.color_palette('Oranges').as_hex()[3],
    'D5': sns.color_palette('Purples').as_hex()[2],
    'D15': sns.color_palette('Purples').as_hex()[5],
    'Spleen': sns.color_palette('Blues').as_hex()[3],
    'Bone Marrow': sns.color_palette('Greens').as_hex()[3],
    'Meninges': sns.color_palette('Reds').as_hex()[3]
}

CBE_BC_COUNTS_ONLY = CBE_BC_COUNTS.iloc[:, 2:].copy()
CBE_EPO_COUNTS_ONLY = CBE_EPO_COUNTS.iloc[:, 2:].copy()

print('Focused CBE guides:', len(CBE))
print('Focused CBE NGG guides:', len(CBE_NGG))
print('Validation dir:', VAL_DIR)
print('Output dir:', FIG_DIR)


In [ ]:
def ecdf(values):
    values = np.asarray(values)
    values = values[~np.isnan(values)]
    x = np.sort(values)
    y = np.arange(1, len(x) + 1) / len(x)
    return x, y


def pca_scatter(data, ax, title, palette):
    corr = data.corr(method='spearman')
    Xt = PCA(n_components=2).fit_transform(corr)
    var = PCA(n_components=2).fit(corr).explained_variance_ratio_
    for i in range(len(Xt)):
        ax.scatter(Xt[i][0], Xt[i][1], edgecolor='black', s=140, alpha=0.85, c=palette[i])
    ax.set_xlabel(f'PC1\n{var[0] * 100:.1f}% variance', fontsize=13)
    ax.set_ylabel(f'PC2\n{var[1] * 100:.1f}% variance', fontsize=13)
    ax.set_title(title, fontsize=14)
    ax.spines[['top', 'right']].set_visible(False)


def z_score_frame(df, mapping):
    out = df.copy()
    for sample in mapping:
        col = f'LFC_median_{sample}'
        out[f'z_score_{sample}'] = (out[col] - out[col].mean()) / out[col].std()
    return out


def add_role_in_cancer(df, gene_col='gene_name_h'):
    role_map = {
        'TSG': 'TSG',
        'TSG, fusion': 'TSG',
        'Undefined': 'Undefined',
        'fusion': 'Undefined',
        'oncogene': 'Oncogene',
        'oncogene, TSG': 'Oncogene/TSG',
        'oncogene, TSG, fusion': 'Oncogene/TSG',
        'oncogene, fusion': 'Oncogene'
    }
    lookup = COSMIC[['Gene Symbol', 'Role in Cancer']].drop_duplicates()
    merged = df.merge(lookup, left_on=gene_col, right_on='Gene Symbol', how='left')
    merged['Role in Cancer'] = merged['Role in Cancer'].fillna('Undefined').map(lambda x: role_map.get(x, 'Undefined'))
    merged = merged.drop(columns=['Gene Symbol'])
    return merged


def load_compact_tables(subdir):
    folder = CRISPRESSO_DIR / subdir
    files = sorted([p for p in folder.glob('*.csv') if p.is_file()])
    return {p.stem.replace('_compact_unfiltered', ''): pd.read_csv(p).rename(columns={'Guide_ID': 'gRNA_id'}) for p in files}


def summarize_editing_by_sample(compact_dict, library_subset, min_sensor_reads=100):
    rows = []
    for sample_name, df in compact_dict.items():
        merged = pd.merge(df, library_subset, on='gRNA_id')
        merged = merged[merged['Reads_aligned_all_amplicons'] >= min_sensor_reads].copy()
        sample_key = sample_name.split('-')[0] if sample_name.startswith('d15') else sample_name.split('rep')[0].rstrip('-')
        if sample_name.startswith('bm'):
            sample_key = 'bm'
        elif sample_name.startswith('spleen'):
            sample_key = 'spleen'
        elif sample_name.startswith('men'):
            sample_key = 'men'
        elif sample_name.startswith('input'):
            sample_key = 'input'
        elif sample_name.startswith('lib'):
            sample_key = 'lib'
        elif sample_name.startswith('d5'):
            sample_key = 'd5'
        elif sample_name.startswith('d15'):
            sample_key = 'd15'
        rows.append({
            'sample_raw': sample_name,
            'Sample': name_dict[sample_key],
            'target_base_edit_perc': merged['target_base_edit_perc'].mean(),
            'corr_perc': merged['corr_perc'].mean(),
            'n_guides': len(merged)
        })
    out = pd.DataFrame(rows)
    out1 = out[['sample_raw', 'Sample', 'n_guides']].copy()
    out1['Editing %'] = out['target_base_edit_perc']
    out1['Edit Type'] = 'Target Editing (w/ Bystanders)'
    out2 = out[['sample_raw', 'Sample', 'n_guides']].copy()
    out2['Editing %'] = out['corr_perc']
    out2['Edit Type'] = 'Pure Correct Editing'
    return pd.concat([out1, out2], ignore_index=True)


def load_mle_tables(subdir):
    folder = MLE_DIR / subdir
    tables = {}
    for fp in sorted(folder.glob('*.csv')):
        tables[fp.stem] = pd.read_csv(fp)
    return tables


def plot_hexbin_corr(df, x_col, y_col, ax, xlabel, ylabel, title):
    rp, _ = scipy.stats.pearsonr(df[x_col], df[y_col])
    rs, _ = scipy.stats.spearmanr(df[x_col], df[y_col])
    ax.hexbin(df[x_col], df[y_col], bins='log', gridsize=22, linewidth=0, extent=(-4, 4, -4, 4))
    ax.plot([-4, 4], [-4, 4], linestyle='dashed', color='black', linewidth=1)
    ax.axhline(0, linestyle='dashed', color='black', linewidth=1)
    ax.axvline(0, linestyle='dashed', color='black', linewidth=1)
    ax.set_aspect('equal')
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(f'{title}\n$R_p$={rp:.2f} | $R_s$={rs:.2f}', fontsize=12)
    ax.spines[['top', 'right']].set_visible(False)


def draw_venn(ax, set1, set2, labels=('A', 'B'), colors=('red', 'tab:blue')):
    if venn2 is not None:
        venn2([set1, set2], set_labels=labels, set_colors=colors, ax=ax)
        return
    only1 = len(set1 - set2)
    only2 = len(set2 - set1)
    both = len(set1 & set2)
    c1 = plt.Circle((0.42, 0.5), 0.23, color=colors[0], alpha=0.35)
    c2 = plt.Circle((0.58, 0.5), 0.23, color=colors[1], alpha=0.35)
    ax.add_patch(c1)
    ax.add_patch(c2)
    ax.text(0.32, 0.5, str(only1), ha='center', va='center', fontsize=12)
    ax.text(0.68, 0.5, str(only2), ha='center', va='center', fontsize=12)
    ax.text(0.50, 0.5, str(both), ha='center', va='center', fontsize=12, fontweight='bold')
    ax.text(0.30, 0.2, labels[0], ha='center', fontsize=11)
    ax.text(0.70, 0.2, labels[1], ha='center', fontsize=11)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')


def build_hit_sets(df, tissues, fdr_suffix='fishers', edit_col='target_base_edit_perc', min_edit=20, min_input=100, guide_filter=None):
    hits = {}
    subset = df.copy()
    if guide_filter is not None:
        subset = subset[subset['gRNA_id'].isin(guide_filter)].copy()
    for tissue in tissues:
        tissue_df = subset[
            (subset['classification'].isin(['targeting guide', 'essential truncation guide']))
            & (subset[edit_col] >= min_edit)
            & (subset['Input_median'] >= min_input)
            & (subset[f'FDR_{tissue}_{fdr_suffix}'] < 0.1)
        ].copy()
        hits[tissue] = set(tissue_df['gRNA_id'])
    return hits


def plot_gene_waterfall(df, gene, tissues, title_prefix, color, edit_col='target_base_edit_perc', min_edit=20, min_input=100):
    subset = df[
        (df['classification'].isin(['targeting guide', 'essential truncation guide']))
        & (df['Gene'] == gene)
        & (df[edit_col] >= min_edit)
        & (df['Input_median'] >= min_input)
    ].copy()
    fig, axes = plt.subplots(1, len(tissues), figsize=(4 * len(tissues), 6), sharey=True)
    if len(tissues) == 1:
        axes = [axes]
    for ax, tissue in zip(axes, tissues):
        rank_df = subset.sort_values(by=f'LFC_median_{tissue}', ascending=False).copy()
        rank_df['Rank'] = np.arange(1, len(rank_df) + 1)
        nonsig = rank_df[rank_df[f'FDR_{tissue}_fishers'] >= 0.1]
        ax.scatter(nonsig['Rank'], nonsig[f'LFC_median_{tissue}'], color='lightgrey', s=12, alpha=0.6)
        levels = [1e-1, 1e-2, 1e-3, 1e-4]
        sizes = {1e-1: 20, 1e-2: 40, 1e-3: 80, 1e-4: 140}
        for idx, cut in enumerate(levels):
            if idx == len(levels) - 1:
                sig = rank_df[rank_df[f'FDR_{tissue}_fishers'] < cut]
            else:
                sig = rank_df[(rank_df[f'FDR_{tissue}_fishers'] < cut) & (rank_df[f'FDR_{tissue}_fishers'] >= levels[idx + 1])]
            ax.scatter(sig['Rank'], sig[f'LFC_median_{tissue}'], color=color, s=sizes[cut], alpha=0.9)
        ax.axhline(0, linestyle='dashed', color='black', linewidth=1)
        ax.set_title(name_dict[tissue], fontsize=12)
        ax.set_xlabel('Rank', fontsize=12)
        ax.spines[['top', 'right']].set_visible(False)
    axes[0].set_ylabel('Median LFC', fontsize=12)
    fig.suptitle(f'{title_prefix} | {gene}', fontsize=14)
    fig.tight_layout()
    return fig


## 1. Counts QC

Replicates, count skew, and guide-count distributions for the new CBE screen.


In [ ]:
def sample_group_from_name(sample):
    if sample.startswith('lib'):
        return 'Plasmid'
    if sample.startswith('input'):
        return 'Input (D0)'
    if sample.startswith('d5'):
        return 'D5'
    if sample.startswith('d15'):
        return 'D15'
    if sample.startswith('spleen'):
        return 'Spleen'
    if sample.startswith('bm'):
        return 'Bone Marrow'
    if sample.startswith('men'):
        return 'Meninges'
    raise ValueError(f'Unknown sample name: {sample}')


bc_sample_groups = [sample_group_from_name(x) for x in CBE_BC_COUNTS_ONLY.columns]
epo_sample_groups = [sample_group_from_name(x) for x in CBE_EPO_COUNTS_ONLY.columns]
palette_bc = [palette_group[g] for g in bc_sample_groups]
palette_epo = [palette_group[g] for g in epo_sample_groups]

sample_summary = pd.DataFrame({
    'BC sample': pd.Series(CBE_BC_COUNTS_ONLY.columns),
    'BC group': pd.Series(bc_sample_groups),
    'EPO sample': pd.Series(CBE_EPO_COUNTS_ONLY.columns),
    'EPO group': pd.Series(epo_sample_groups)
})
sample_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
pca_scatter(CBE_BC_COUNTS_ONLY, axes[0], 'CBE BC counts PCA', palette_bc)
pca_scatter(CBE_EPO_COUNTS_ONLY, axes[1], 'CBE EPO counts PCA', palette_epo)
fig.tight_layout()
fig.savefig(FIG_DIR / 'cbe_counts_pca.pdf', bbox_inches='tight')
plt.show()


In [ ]:
def skew_ratio_frame(counts_only, kind):
    rows = []
    for sample in counts_only.columns:
        vals = np.sort(counts_only[sample].to_numpy())
        q10 = vals[int(0.1 * len(vals))]
        q90 = vals[int(0.9 * len(vals))]
        ratio = q90 / q10 if q10 > 0 else q90
        if sample.startswith('lib'):
            label = 'Plasmid'
        elif sample.startswith('input'):
            label = 'Input (D0)'
        elif sample.startswith('d5'):
            label = 'D5'
        elif sample.startswith('d15'):
            label = 'D15'
        elif sample.startswith('spleen'):
            label = 'Spleen'
        elif sample.startswith('bm'):
            label = 'Bone Marrow'
        elif sample.startswith('men'):
            label = 'Meninges'
        rows.append({'Screen': kind, 'Sample': sample, 'Group': label, '90/10 Skew Ratio': ratio})
    return pd.DataFrame(rows)

skew_df = pd.concat([
    skew_ratio_frame(CBE_BC_COUNTS_ONLY, 'BC'),
    skew_ratio_frame(CBE_EPO_COUNTS_ONLY, 'EPO')
], ignore_index=True)

order = ['Plasmid', 'Input (D0)', 'D5', 'D15', 'Spleen', 'Bone Marrow', 'Meninges']
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharex=True, sharey=True)
for ax, screen in zip(axes, ['BC', 'EPO']):
    plot_df = skew_df[skew_df['Screen'] == screen]
    sns.barplot(data=plot_df, y='Group', x='90/10 Skew Ratio', order=order, palette=palette_group, edgecolor='black', linewidth=1, ax=ax)
    sns.stripplot(data=plot_df, y='Group', x='90/10 Skew Ratio', order=order, palette=palette_group, edgecolor='black', linewidth=0.8, size=7, ax=ax)
    ax.set_title(f'CBE {screen}', fontsize=13)
    ax.set_ylabel('')
    ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / 'cbe_skew_ratio.pdf', bbox_inches='tight')
plt.show()


In [ ]:
def sample_titles(counts_only):
    labels = []
    for sample in counts_only.columns:
        labels.append(sample)
    return labels

for screen_name, counts_only, color, xmax, nrows in [
    ('BC', CBE_BC_COUNTS_ONLY, sns.color_palette('Blues').as_hex()[2], 50000, 3),
    ('EPO', CBE_EPO_COUNTS_ONLY, sns.color_palette('Blues').as_hex()[5], 50000, 4),
]:
    fig, axes = plt.subplots(nrows, 10, figsize=(20, 2.2 * nrows), sharex=True, sharey=False)
    axes = np.array(axes).reshape(nrows, 10)
    bins = np.linspace(0, xmax, 81)
    for idx, sample in enumerate(counts_only.columns):
        ax = axes[idx // 10, idx % 10]
        ax.hist(np.clip(counts_only[sample], bins[0], bins[-1]), bins=bins, color=color, edgecolor='black', linewidth=0)
        ax.set_title(sample, fontsize=10)
        ax.spines[['top', 'right']].set_visible(False)
    for idx in range(len(counts_only.columns), nrows * 10):
        fig.delaxes(axes[idx // 10, idx % 10])
    fig.suptitle(f'CBE {screen_name} guide-count distributions', fontsize=14)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'cbe_{screen_name.lower()}_count_histograms.pdf', bbox_inches='tight')
    plt.show()


## 2. Editing QC

Average editing by sample, MLE-level editing concordance, and D5 editing distributions.


In [ ]:
compact_bc = load_compact_tables('CBE_BC')
compact_epo = load_compact_tables('CBE_EPO')

editing_bc_all = summarize_editing_by_sample(compact_bc, CBE, min_sensor_reads=100)
editing_epo_all = summarize_editing_by_sample(compact_epo, CBE, min_sensor_reads=100)
editing_bc_ngg = summarize_editing_by_sample(compact_bc, CBE_NGG, min_sensor_reads=100)
editing_epo_ngg = summarize_editing_by_sample(compact_epo, CBE_NGG, min_sensor_reads=100)

order = ['Plasmid', 'Input (D0)', 'D5', 'D15', 'Spleen', 'Bone Marrow', 'Meninges']
fig, axes = plt.subplots(1, 2, figsize=(10, 6), sharex=True, sharey=True)
for ax, plot_df, title in zip(
    axes,
    [editing_bc_ngg, editing_epo_ngg],
    ['CBE BC (NGG only)', 'CBE EPO (NGG only)']
):
    sns.barplot(data=plot_df, y='Sample', x='Editing %', hue='Edit Type', order=order, edgecolor='black', linewidth=1, palette=['tab:blue', 'tab:purple'], ax=ax)
    sns.stripplot(data=plot_df, y='Sample', x='Editing %', hue='Edit Type', order=order, dodge=True, palette=['tab:blue', 'tab:purple'], edgecolor='black', linewidth=0.5, size=5, ax=ax)
    ax.set_title(title, fontsize=13)
    ax.set_ylabel('')
    ax.spines[['top', 'right']].set_visible(False)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[:2], labels[:2], frameon=False, fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / 'cbe_average_editing_by_sample.pdf', bbox_inches='tight')
plt.show()


In [ ]:
mle_cbe_epo = load_mle_tables('CBE_EPO')
editing_cols = {}
read_cols = {}
sample_name_map = {
    'input': 'INPUT',
    'd15': 'D15',
    'd5': 'D5',
    'lib': 'PLASMID',
    'spleen': 'S',
    'men': 'M',
    'bm': 'BM'
}

for key, df in mle_cbe_epo.items():
    merged = pd.merge(CBE, df.rename(columns={'Guide_ID': 'gRNA_id'}), on='gRNA_id')
    screen_key = sample_name_map.get(key, key.upper())
    editing_cols[screen_key] = merged['target_base_edit_perc']
    read_cols[f'reads_{key}'] = merged['Reads_aligned_all_amplicons']

editing_df = pd.DataFrame(editing_cols)
reads_df = pd.DataFrame(read_cols)
editing_combined = pd.concat([editing_df, reads_df], axis=1)
editing_filtered = editing_combined[
    (editing_combined['reads_input'] >= 10)
    & (editing_combined['reads_spleen'] >= 10)
    & (editing_combined['reads_men'] >= 10)
    & (editing_combined['reads_bm'] >= 10)
].copy()

g = sns.clustermap(editing_filtered[['INPUT', 'D15', 'D5', 'PLASMID', 'S', 'M', 'BM']].corr(method='pearson'), cmap='Reds', vmin=0, annot=True, linewidth=1, annot_kws={'size': 12})
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xmajorticklabels(), fontsize=12)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_ymajorticklabels(), fontsize=12, rotation=0)
g.fig.suptitle('CBE editing correlation (MLE, EPO)', y=1.02)
g.savefig(FIG_DIR / 'cbe_editing_correlation_heatmap.pdf', bbox_inches='tight')
plt.show()


In [ ]:
d5_cbe = pd.read_csv(MLE_DIR / 'CBE_EPO' / 'd5.csv').rename(columns={'Guide_ID': 'gRNA_id'})
cbe_d5_merged = pd.merge(d5_cbe, CBE_NGG, on='gRNA_id')
cbe_d5_filtered = cbe_d5_merged[(cbe_d5_merged['Reads_in_input'] >= 100) & (cbe_d5_merged['Reads_aligned_all_amplicons'] >= 10)].copy()

pam_groups = {
    'NGG': cbe_d5_filtered[cbe_d5_filtered['PAM_NGN'].isin([f'{x}GG' for x in ['A', 'T', 'C', 'G']])],
    'NGA': cbe_d5_filtered[cbe_d5_filtered['PAM_NGN'].isin([f'{x}GA' for x in ['A', 'T', 'C', 'G']])],
    'NGT': cbe_d5_filtered[cbe_d5_filtered['PAM_NGN'].isin([f'{x}GT' for x in ['A', 'T', 'C', 'G']])],
    'NGC': cbe_d5_filtered[cbe_d5_filtered['PAM_NGN'].isin([f'{x}GC' for x in ['A', 'T', 'C', 'G']])]
}

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for label, color in zip(['NGG', 'NGA', 'NGT', 'NGC'], sns.color_palette('Set2', 4)):
    x, y = ecdf(pam_groups[label]['target_base_edit_perc'])
    ax[0].plot(x, y, linewidth=3, label=f'{label} (n={len(pam_groups[label])})', color=color)
ax[0].set_xlabel('Target base editing %', fontsize=12)
ax[0].set_ylabel('CDF', fontsize=12)
ax[0].set_title('CBE D5 editing by PAM', fontsize=13)
ax[0].legend(frameon=False, fontsize=10)
ax[0].spines[['top', 'right']].set_visible(False)

ax[1].hist([
    pam_groups['NGG']['target_base_edit_perc'].to_numpy(),
    pd.concat([pam_groups['NGA'], pam_groups['NGT'], pam_groups['NGC']], ignore_index=True)['target_base_edit_perc'].to_numpy()
], bins=np.linspace(0, 100, 51), stacked=True, edgecolor='black', linewidth=1, color=[sns.color_palette('Oranges').as_hex()[3], sns.color_palette('Greens').as_hex()[2]], label=[f'NGG (n={len(pam_groups["NGG"])})', 'NGN non-NGG'])
ax[1].set_xlabel('Target base editing %', fontsize=12)
ax[1].set_ylabel('No. of gRNAs', fontsize=12)
ax[1].set_title('CBE D5 NGG vs non-NGG', fontsize=13)
ax[1].legend(frameon=False, fontsize=10)
ax[1].spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / 'cbe_d5_pam_editing.pdf', bbox_inches='tight')
plt.show()


## 3. LFC correlations

Correlations across tissues and across focused/input-baseline, focused/BC-baseline, and OG/focused comparisons.


In [ ]:
focused_samples = ['bm', 'd15', 'd5', 'men', 'spleen']
og_samples = ['bonemarrow', 'd15', 'd5', 'meninges', 'spleen']

LFC_CBE_Z = z_score_frame(LFC_CBE, focused_samples)
LFC_CBE_VS_BC_Z = z_score_frame(LFC_CBE_VS_BC, focused_samples)
LFC_CBE_OG_Z = z_score_frame(LFC_CBE_OG, og_samples)

LFC_CBE_NGG = LFC_CBE_Z[LFC_CBE_Z['gRNA_id'].isin(GUIDES_NGG_CBE_FOCUSED)].reset_index(drop=True)
g = sns.clustermap(LFC_CBE_NGG[[f'LFC_median_{x}' for x in focused_samples]].corr(method='pearson'), cmap='Reds', vmin=0, annot=True, linewidth=1, annot_kws={'size': 12})
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xmajorticklabels(), fontsize=12)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_ymajorticklabels(), fontsize=12, rotation=0)
g.fig.suptitle('CBE focused LFC correlation (NGG only)', y=1.02)
g.savefig(FIG_DIR / 'cbe_focused_lfc_correlation.pdf', bbox_inches='tight')
plt.show()


In [ ]:
df_focus_vs_bc = LFC_CBE_NGG[['gRNA_id'] + [f'z_score_{x}' for x in focused_samples]].copy()
df_focus_vs_bc.columns = ['gRNA_id'] + [f'focused_{x}' for x in focused_samples]
tmp_bc = LFC_CBE_VS_BC_Z[LFC_CBE_VS_BC_Z['gRNA_id'].isin(GUIDES_NGG_CBE_FOCUSED)][['gRNA_id'] + [f'z_score_{x}' for x in focused_samples]].copy()
tmp_bc.columns = ['gRNA_id'] + [f'vs_bc_{x}' for x in focused_samples]
df_focus_vs_bc = df_focus_vs_bc.merge(tmp_bc, on='gRNA_id')

tmp_og = LFC_CBE_OG_Z[LFC_CBE_OG_Z['gRNA_id'].isin(GUIDES_NGG_CBE_OG)][['gRNA_id'] + [f'z_score_{x}' for x in og_samples]].copy()
tmp_og.columns = ['gRNA_id'] + ['og_bm', 'og_d15', 'og_d5', 'og_men', 'og_spleen']
df_og_vs_focused = df_focus_vs_bc.merge(tmp_og, on='gRNA_id')

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, tissue in enumerate(['d15', 'bm', 'spleen', 'men']):
    plot_hexbin_corr(df_focus_vs_bc, f'vs_bc_{tissue}', f'focused_{tissue}', axes[0, i], 'Z-score BC baseline', 'Z-score input baseline', name_dict[tissue])
    plot_hexbin_corr(df_og_vs_focused, f'og_{tissue}', f'focused_{tissue}', axes[1, i], 'Z-score OG', 'Z-score focused', name_dict[tissue])
axes[0, 0].set_ylabel('Z-score input baseline', fontsize=12)
axes[1, 0].set_ylabel('Z-score focused', fontsize=12)
fig.suptitle('CBE LFC correlation comparisons', fontsize=15)
fig.tight_layout()
fig.savefig(FIG_DIR / 'cbe_lfc_hexbin_correlations.pdf', bbox_inches='tight')
plt.show()


## 4. Hit overlap analyses

Focused T0 vs BC baseline, and focused vs OG hit overlap, using the same main cutoffs you used elsewhere.


In [ ]:
focused_hits = build_hit_sets(
    LFC_CBE,
    tissues=['d15', 'spleen', 'bm', 'men'],
    fdr_suffix='fishers',
    edit_col='target_base_edit_perc',
    min_edit=20,
    min_input=100,
    guide_filter=GUIDES_NGG_CBE_FOCUSED
)

bc_hits = build_hit_sets(
    LFC_CBE_VS_BC,
    tissues=['d15', 'spleen', 'bm', 'men'],
    fdr_suffix='fishers',
    edit_col='target_base_edit_perc_epo',
    min_edit=20,
    min_input=100,
    guide_filter=GUIDES_NGG_CBE_FOCUSED
)

og_hits = build_hit_sets(
    LFC_CBE_OG,
    tissues=['d15', 'spleen', 'bonemarrow', 'meninges'],
    fdr_suffix='fishers',
    edit_col='target_base_edit_perc',
    min_edit=20,
    min_input=10,
    guide_filter=GUIDES_NGG_CBE_OG
)

fig, axes = plt.subplots(1, 4, figsize=(12, 3.5))
for ax, tissue in zip(axes, ['d15', 'spleen', 'bm', 'men']):
    draw_venn(ax, focused_hits[tissue], bc_hits[tissue], labels=('T0', 'BC'), colors=('red', 'tab:blue'))
    ax.set_title(name_dict[tissue], fontsize=12)
fig.suptitle('CBE focused hits: T0 vs BC baseline\nFishers FDR < 0.1, editing >= 20%, NGG only', fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / 'cbe_hits_t0_vs_bc.pdf', bbox_inches='tight')
plt.show()

og_map = {'d15': 'd15', 'spleen': 'spleen', 'bm': 'bonemarrow', 'men': 'meninges'}
fig, axes = plt.subplots(1, 4, figsize=(12, 3.5))
for ax, tissue in zip(axes, ['d15', 'spleen', 'bm', 'men']):
    draw_venn(ax, focused_hits[tissue], og_hits[og_map[tissue]] & set(LFC_CBE['gRNA_id']), labels=('Focused', 'OG'), colors=('red', 'tab:blue'))
    ax.set_title(name_dict[tissue], fontsize=12)
fig.suptitle('CBE hits: focused vs OG\nFishers FDR < 0.1, editing >= 20%, NGG only', fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / 'cbe_hits_focused_vs_og.pdf', bbox_inches='tight')
plt.show()


In [ ]:
df_heat = add_role_in_cancer(LFC_CBE.copy())
heat_rows = []
for tissue in ['d15', 'spleen', 'bm', 'men']:
    top_hits = df_heat[
        (df_heat['classification'] == 'targeting guide')
        & (df_heat['target_base_edit_perc'] >= 20)
        & (df_heat['Input_median'] >= 100)
        & (df_heat[f'FDR_{tissue}_fishers'] < 0.1)
        & (df_heat['gRNA_id'].isin(GUIDES_NGG_CBE_FOCUSED))
    ].sort_values(by=f'LFC_median_{tissue}', ascending=False).head(10).copy()
    if len(top_hits) == 0:
        continue
    tmp = top_hits[['Gene', 'HGVSp_m', 'gRNA_id', 'LFC_median_d15', 'LFC_median_spleen', 'LFC_median_bm', 'LFC_median_men']].copy()
    tmp['selected_in'] = tissue
    heat_rows.append(tmp)

heat_df = pd.concat(heat_rows, ignore_index=True).drop_duplicates(subset=['gRNA_id'])
heat_df = heat_df.set_index(['Gene', 'HGVSp_m'])[['LFC_median_d15', 'LFC_median_spleen', 'LFC_median_bm', 'LFC_median_men']]
fig, ax = plt.subplots(figsize=(8, max(5, 0.28 * len(heat_df))))
sns.heatmap(heat_df, cmap='seismic', center=0, linewidth=0.5, edgecolor='white', ax=ax)
ax.set_title('Top CBE enrichers across tissues\n(Focused, Fishers FDR < 0.1, editing >= 20%, NGG only)', fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / 'cbe_top_hit_heatmap.pdf', bbox_inches='tight')
plt.show()


## 5. Gene-level waterfall plots

Waterfall/rank-style plots for the main genes you repeatedly inspect in focused CBE.


In [ ]:
for gene, color in [('Trp53', sns.color_palette('Blues').as_hex()[2]), ('Pax5', sns.color_palette('Purples').as_hex()[2]), ('Ctnnb1', sns.color_palette('Greens').as_hex()[3])]:
    fig = plot_gene_waterfall(
        LFC_CBE[LFC_CBE['gRNA_id'].isin(GUIDES_NGG_CBE_FOCUSED)].copy(),
        gene=gene,
        tissues=['d15', 'spleen', 'bm', 'men'],
        title_prefix='CBE focused (T0 baseline)',
        color=color,
        edit_col='target_base_edit_perc',
        min_edit=20,
        min_input=100
    )
    fig.savefig(FIG_DIR / f'cbe_waterfall_{gene.lower()}_t0.pdf', bbox_inches='tight')
    plt.show()

for gene, color in [('Trp53', sns.color_palette('Blues').as_hex()[4]), ('Pax5', sns.color_palette('Purples').as_hex()[4]), ('Ctnnb1', sns.color_palette('Greens').as_hex()[5])]:
    fig = plot_gene_waterfall(
        LFC_CBE_VS_BC[LFC_CBE_VS_BC['gRNA_id'].isin(GUIDES_NGG_CBE_FOCUSED)].copy(),
        gene=gene,
        tissues=['d15', 'spleen', 'bm', 'men'],
        title_prefix='CBE focused (BC baseline)',
        color=color,
        edit_col='target_base_edit_perc_epo',
        min_edit=20,
        min_input=100
    )
    fig.savefig(FIG_DIR / f'cbe_waterfall_{gene.lower()}_bc.pdf', bbox_inches='tight')
    plt.show()


## 6. Quick exports

This table is convenient for downstream manual review.


In [ ]:
review_cols = [
    'gRNA_id', 'Gene', 'gene_name_h', 'HGVSp_m', 'HGVSp_h', 'classification',
    'Input_median', 'sensor_reads', 'target_base_edit_perc', 'corr_perc',
    'LFC_median_d15', 'LFC_median_spleen', 'LFC_median_bm', 'LFC_median_men',
    'FDR_d15_fishers', 'FDR_spleen_fishers', 'FDR_bm_fishers', 'FDR_men_fishers'
]
export_df = LFC_CBE[LFC_CBE['gRNA_id'].isin(GUIDES_NGG_CBE_FOCUSED)][review_cols].copy()
export_df.to_csv(FIG_DIR / 'cbe_focused_review_table.csv', index=False)
export_df.head()
